In [ ]:
import pandas as pd
import os
import json
from langchain_google_vertexai import VertexAI
from vertexai import generative_models
from tqdm import tqdm

path_to_creds = r"vertex-ai_creds.json"
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = json.load(open(path_to_creds))["path"]

In [ ]:
# Safety config
safety_config = {
    generative_models.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: generative_models.HarmBlockThreshold.BLOCK_ONLY_HIGH ,
    generative_models.HarmCategory.HARM_CATEGORY_HARASSMENT: generative_models.HarmBlockThreshold.BLOCK_ONLY_HIGH,
    generative_models.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: generative_models.HarmBlockThreshold.BLOCK_ONLY_HIGH,
    generative_models.HarmCategory.HARM_CATEGORY_HATE_SPEECH: generative_models.HarmBlockThreshold.BLOCK_ONLY_HIGH,
}

In [ ]:
model = VertexAI(model_name="gemini-1.5-pro-preview-0409",max_output_tokens=3000, safety_settings=safety_config, temperature=0)

In [ ]:
df = pd.read_csv('reddit_posts_filtered_new.csv') ### different file than main analysis

In [ ]:
from pydantic import BaseModel, Field

enums_answers_text = [
    "yes",
    "no",
    "plausibly",
    "cannot be inferred"
]
class RiskFactors(BaseModel):
    children: str = Field(
        enum=enums_answers_text,
        description="Presence of children shared between the offender and the victim"
    )
    emotional_violence: str = Field(
        enum=enums_answers_text,
        description="Emotional or psychological harm inflicted by the offender on the victim"
    )
    physical_violence: str = Field(
        enum=enums_answers_text,
        description="Physical pain or harm inflicted by the offender on the victim"
    )
    sexual_violence: str = Field(
        enum=enums_answers_text,
        description="Non-consensual sexual acts performed by the offender on the victim"
    )
    economic_violence: str = Field(
        enum=enums_answers_text,
        description="Control exerted by the offender over the victim's financial resources"
    )
    past_offenses: str = Field(
        enum=enums_answers_text,
        description="Any historical criminal or abusive behavior by the offender"
    )
    social_isolation: str = Field(
        enum=enums_answers_text,
        description="Restriction of the victim’s social contacts and interactions by the offender"
    )
    refuses_treatment: str = Field(
        enum=enums_answers_text,
        description="The offender’s unwillingness to seek or continue treatment for behavioral or mental health issues"
    )
    suicidal_threats: str = Field(
        enum=enums_answers_text,
        description="Threats of suicide made by the offender, often as a manipulative tactic"
    )
    mental_condition: str = Field(
        enum=enums_answers_text,
        description="Presence of diagnosed mental health conditions affecting the offender"
    )
    daily_activity_control: str = Field(
        enum=enums_answers_text,
        description="Control over the victim’s daily life and decisions by the offender"
    )
    violent_behavior: str = Field(
        enum=enums_answers_text,
        description="Does the offender exhibit violent behavior towards others?"
    )
    unemployment: str = Field(
        enum=enums_answers_text,
        description="The offender is unemployed"
    )
    substance_use: str = Field(
        enum=enums_answers_text,
        description="Dependency on drugs, alcohol, or other harmful substances by the offender."
    )
    obsessiveness: str = Field(
        enum=enums_answers_text,
        description="The offender’s excessive fixation on the victim, often controlling or overbearing"
    )
    jealousy: str = Field(
        enum=enums_answers_text,
        description="Displays of jealousy by the offender, potentially leading to controlling behavior."
    )
    outbursts: str = Field(
        enum=enums_answers_text,
        description="Sudden and intense episodes of anger exhibited by the offender."
    )
    ptsd: str = Field(
        enum=enums_answers_text,
        description="The offender suffers from post-traumatic stress disorder"
    )
    hard_childhood: str = Field(
        enum=enums_answers_text,
        description="The offender suffers from a traumatic childhood experience"
    )
    emotional_dependency: str = Field(
        enum=enums_answers_text,
        description="The offender’s emotional or psychological dependence on the victim."
    )
    prevention_of_care: str = Field(
        enum=enums_answers_text,
        description="The offender's denial of the need for medical or mental care"
    )
    fear_based_relationship: str = Field(
        enum=enums_answers_text,
        description="The victim experiences fear within the relationship."
    )
    humiliation: str = Field(
        enum=enums_answers_text,
        description="Actions by the offender that humiliate or demean the victim"
    )
    physical_threats: str = Field(
        enum=enums_answers_text,
        description="Threats of physical violence made by the offender towards the victim"
    )
    presence_of_others_in_assault: str = Field(
        enum=enums_answers_text,
        description="Presence of others during acts of assault or violence initiated by the offender."
    )
    signs_of_injury: str = Field(
        enum=enums_answers_text,
        description="Physical signs of injury on the victim indicating abuse."
    )
    property_damage: str = Field(
        enum=enums_answers_text,
        description="Damage to property by the offender as a form of intimidation or aggression"
    )
    access_to_weapons: str = Field(
        enum=enums_answers_text,
        description="The offender’s access to weapons, posing a potential risk of violent behavior."
    )
    gaslighting: str = Field(
        enum=enums_answers_text,
        description="Psychological manipulation by the offender causing the victim to doubt their perception"
    )
    victim_support_network: str = Field(
        enum=enums_answers_text,
        description="Availability of a social support network (such as family or friends) for the victim"
    )
    offender_support_network: str = Field(
        enum=enums_answers_text,
        description="Availability of a social support network (such as family or friends) for the offender"
    )
    attempts_to_end_relationship: str = Field(
        enum=enums_answers_text,
        description="Attempts by the victim to end the relationship"
    )
    refusal_to_end_relationship: str = Field(
        enum=enums_answers_text,
        description="Refusal by the offender to accept the end of the relationship"
    )
    public_private_discrepancy: str = Field(
        enum=enums_answers_text,
        description="Discrepancy between the offender’s public persona and private behavior"
    )
    narcissistic_traits: str = Field(
        enum=enums_answers_text,
        description="Exhibitions of narcissism by the offender, including grandiosity and lack of empathy"
    )
    aggressive_behavior: str = Field(
        enum=enums_answers_text,
        description="The offender's tendency to behave violently or aggressively."
    )


In [ ]:
prompt_template_str_risk = """
Evaluate the potential for danger in a relationship as described in a Reddit post by assessing the presence of various risk factors. Provide responses of 'yes', 'no', 'plausibly', 'cannot be inferred' for each risk factor, based only on details written in the story.

Use the following guidelines:

Yes - Clear evidence in the text supports the presence of the risk factor.
No - Clear evidence in the text indicates the absence of the risk factor.
Plausibly - There is some indirect or vague evidence suggesting the presence of the risk factor, but it is not definitive.
Cannot Be Inferred - There is insufficient information in the text to make a determination about the presence or absence of the risk factor.

Story: {story}

Format:
{format_instructions}
"""

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser_risk = PydanticOutputParser(pydantic_object=RiskFactors)
format_instructions_risk = parser_risk.get_format_instructions()
prompt_risk = PromptTemplate(
    template=prompt_template_str_risk,
    input_variables=["story"],
    partial_variables={"format_instructions": format_instructions_risk}
)
chain_risk = prompt_risk | model | parser_risk

In [ ]:
post = """
TL;DR My boyfriend has depression which paralyzes him, despite medication and therapy. We’ve been in a big rut in our lives, emotionally co-dependent, etc. I want to change things and get us healthy but I have a fair amount of repressed anger about his paralysis that I’m having a hard time processing. If I told him the extent of my feelings it would send him into a depressive spiral so I opt to coddle him instead. Also not healthy, something needs to change. How do I stop bottling my feelings and communicate healthfully without hurting him? I love him and want to fight for him. I won’t consider breaking up as an option yet. Thank you!
"""

In [ ]:
output = chain_risk.invoke({"story": post})

In [ ]:
# Organize attributes by their values
categories = {'yes': [], 'plausibly': [], 'no': []}
for field_name, value in output.dict().items():
    if value not in categories.keys():
        continue
    categories[value].append(field_name.replace('_', ' ').capitalize())

# Printing the results grouped by category
print("Risk Factors Evaluation:")
print("------------------------")
for category, factors in categories.items():
    if factors:  # Only print categories with items
        print(f"\n{category.capitalize()}:")
        for factor in factors:
            print(f"- {factor}")

In [ ]:
sampled_df = df.copy()

In [ ]:
import pickle

output_file = "results_posts_risks.pkl"
results = pickle.load(open(output_file, "rb"))

In [ ]:
existing_results = [res["index"] for res in results]

In [ ]:
len(set(existing_results))

8590

In [ ]:
output_file = "results_posts_risks.pkl"

In [ ]:
import pickle

for i, row in tqdm(sampled_df.iterrows(), total=sampled_df.shape[0]):
    story = row["post_body"]
    if i in existing_results:
        continue
    try:
        output = chain_risk.invoke({"story" : story})
        output_json = json.loads(output.json())
        results.append({"index": i, "story": story, "output": output, "output_json": output_json})
    except Exception as e:
        results.append({"index": i, "story": story, "output": "problem", "output_json": None})
    finally:
        if len(results) % 5 == 0:
            with open(output_file, "wb") as f:
                pickle.dump(results, f)

100%|██████████| 8594/8594 [00:51<00:00, 168.21it/s]  


In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(results)
expanded_json_df = pd.json_normalize(results_df['output_json'])
expanded_json_df["index"] = results_df["index"]
expanded_json_df = expanded_json_df.set_index("index")

In [ ]:
expanded_json_df

In [ ]:
results_df

In [ ]:
results_df = results_df.set_index("index")
results_df = results_df[results_df["output"]!="problem"]
results_df_joined = results_df.join(expanded_json_df, how='inner')
results_df_joined = results_df_joined.drop(["story", "output", "output_json"], axis=1)
final_df_joined = sampled_df.merge(results_df_joined, left_index=True, right_index=True, how='inner')
final_df_joined.shape

(5856, 59)

In [ ]:
final_df_joined.tail(1)

,post_id,title,post_author,body,date_time,url,score,num_comments,flairs,highest_comment_id,...,property_damage,access_to_weapons,gaslighting,victim_support_network,offender_support_network,attempts_to_end_relationship,refusal_to_end_relationship,public_private_discrepancy,narcissistic_traits,aggressive_behavior
8593,1bwvtz5,How do I (F28) process my anger about my boyfr...,PutTheCornBreadDown,I (F28) need serious guidance about my depress...,05/04/24 22:55:26,https://www.reddit.com/r/relationships/comment...,1,6,[],ky9f439,...,cannot be inferred,cannot be inferred,cannot be inferred,cannot be inferred,cannot be inferred,no,cannot be inferred,cannot be inferred,cannot be inferred,no


In [ ]:
final_df_joined.tail(1)["post_body"].values[0]

'How do I (F28) process my anger about my boyfriend’s (M29) depression as someone who has never had depression and can’t empathize no matter how much I want to? Help! \nI (F28) need serious guidance about my depressed boyfriend (M29), who is stuck in a rut. How do I get him out? \n\nThis may be discordant, forgive me. My mind is racing at the moment. \n\nMy boyfriend, Dean, has depression and anxiety, has for years. I have never had either (bless the fucking lord, I won the genetic mental health lottery, I don’t take it for granted) which makes it incredibly difficult for me to empathize with his plight. I can sympathize, and I adore this man so I have lots of compassion for him. But my patience is running thin, this is really wearing me down, and it’s time for me to ask for help, hence why I’m here. \n\nWe graduated from the same program with MA’s right in the heat of covid in 2021 so no one was hiring and we lost our momentum. We haven’t been able to find work in our field, ever. \n\

#### Add metadata

In [ ]:
from typing import Optional

class Labels(BaseModel):
    relationship_status: str = Field(
        enum=["together", "separated", "process of separation", "cannot be inferred", "irrelevant"],
        description="Status of the romantic relationship - together, separated, in the process of separation, cannot be inferred or irrelevant (if not romantic)"
    )
    relationship_type: str = Field(
        description="Describes the nature of the relationship between the individuals mentioned in the post.",
        enum=["friends", "married", "exes", "dating", "separated", "family", "co-workers", "other"]
    )
    living_with: str = Field(
        description="Indicates whether the individuals involved in the relationship are currently living together. If no information about living situation, then answer with cannot be inferred",
        enum=["yes", "no", "cannot be inferred"]
    )
    age_female: Optional[int] = Field(
        default=None,
        description="Specifies the age of the female involved in the scenario described in the post. "
    )
    age_male: Optional[int] = Field(
        default=None,
        description="Specifies the age of the male involved in the scenario described in the post, if applicable"
    )
    author_gender: str = Field(
        description="Denotes the gender of the post's author.",
        enum=["female", "male"]
    )


In [ ]:
prompt_template_str_labels = """
Analyze reddit post and extract information about the relationship.

Story: {story}

{format_instructions}
"""

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

parser_labels = PydanticOutputParser(pydantic_object=Labels)
format_instructions_labels = parser_labels.get_format_instructions()
prompt_labels = PromptTemplate(
    template=prompt_template_str_labels,
    input_variables=["story"],
    partial_variables={"format_instructions": format_instructions_labels}
)
chain_labels = prompt_labels | model | parser_labels

In [ ]:
output_file_labels = "results_posts_labels.pkl"
if os.path.exists(output_file_labels):
    results_labels = pickle.load(open(output_file_labels, "rb"))
else:
    results_labels = []

In [ ]:
output_file_labels = "results_posts_labels.pkl"

In [ ]:
existing_results_labels = [res["index"] for res in results_labels]

In [ ]:
for i, row in tqdm(final_df_joined.iterrows(), total=final_df_joined.shape[0]):
    story = row["post_body"]
    if i in existing_results_labels:
        continue
    try:
        output = chain_labels.invoke({"story" : story})
        output_json = json.loads(output.json())
        results_labels.append({"index": i, "story": story, "output": output, "output_json": output_json})
    except Exception as e:
        print(f"problem with post! - {i} - {e}")
        results_labels.append({"index": i, "story": story, "output": "problem", "output_json": None})
    finally:
        if len(results_labels) % 5 == 0:
            with open(output_file_labels, "wb") as f:
                pickle.dump(results_labels, f)

 18%|█▊        | 1028/5856 [01:02<07:19, 11.00it/s] 

In [ ]:
# Convert results to DataFrame
results_df_labels = pd.DataFrame(results_labels)
results_df_labels = results_df_labels[results_df_labels["output"]!="problem"]
expanded_json_df_labels = pd.json_normalize(results_df_labels['output_json'])
expanded_json_df_labels["index"] = results_df_labels["index"].astype(int)
expanded_json_df_labels = expanded_json_df_labels.set_index("index")
results_df_labels = results_df_labels.set_index("index")

results_df_joined_labels = results_df_labels.join(expanded_json_df_labels, how='inner')
results_df_joined_labels = results_df_joined_labels.drop(["story", "output", "output_json"], axis=1)
final_df_joined_all = final_df_joined.merge(results_df_joined_labels, left_index=True, right_index=True, how='inner')
final_df_joined_all.shape

In [ ]:
final_df_joined.to_csv("reddit_posts_filtered_labeled_risks.csv", index=False)